# Zomato Delivery Time Prediction

**Model:** Random Forest Regression  
**Encoding:** Label Encoding  
**Dashboard:** Power BI

**Flow:** Dataset → Cleaning → Feature Engineering → Label Encoding → Random Forest → Predictions → Power BI

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load Dataset

In [ ]:
df = pd.read_csv("../data/zomato_delivery_dataset.csv")

print(df.shape)
df.head()

## 3. Understand the Data

In [ ]:
print(df.info())
print(df.isnull().sum())
df.describe()

## 4. Remove Duplicates

In [ ]:
df = df.drop_duplicates()
print("Rows:", len(df))

## 5. Date Features

In [ ]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

df["Order_Day"] = df["Order_Date"].dt.day
df["Order_Month"] = df["Order_Date"].dt.month
df["Order_DayOfWeek"] = df["Order_Date"].dt.dayofweek

## 6. Time Features

In [ ]:
def time_to_minutes(x):
    if pd.isna(x):
        return np.nan
    h, m = map(int, str(x).split(":"))
    return h * 60 + m

df["Order_Time"] = df["Time_Orderd"].apply(time_to_minutes)
df["Pickup_Time"] = df["Time_Order_picked"].apply(time_to_minutes)

df["Pickup_Delay"] = (
    df["Pickup_Time"] - df["Order_Time"]
).abs()

## 7. Distance

In [ ]:
def distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )

    a = (
        np.sin((lat2-lat1)/2)**2 +
        np.cos(lat1) * np.cos(lat2) *
        np.sin((lon2-lon1)/2)**2
    )

    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["Distance_km"] = distance(
    df["Restaurant_latitude"],
    df["Restaurant_longitude"],
    df["Delivery_location_latitude"],
    df["Delivery_location_longitude"]
)

## 8. Simple EDA

In [ ]:
sns.histplot(df["Time_taken (min)"], kde=True)
plt.title("Delivery Time Distribution")
plt.show()

In [ ]:
df.groupby("Road_traffic_density")["Time_taken (min)"].mean().plot(kind="bar")
plt.title("Average Delivery Time by Traffic")
plt.ylabel("Minutes")
plt.show()

## 9. Select Features

In [ ]:
features = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Distance_km",
    "Order_Day",
    "Order_Month",
    "Order_DayOfWeek",
    "Order_Time",
    "Pickup_Time",
    "Pickup_Delay",
    "Weather_conditions",
    "Road_traffic_density",
    "Vehicle_condition",
    "Type_of_order",
    "Type_of_vehicle",
    "multiple_deliveries",
    "Festival",
    "City"
]

X = df[features]
y = df["Time_taken (min)"]

## 10. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

## 11. Fill Missing Values

In [ ]:
numeric = X_train.select_dtypes(include=np.number).columns
categorical = X_train.select_dtypes(exclude=np.number).columns

for col in numeric:
    X_train[col] = X_train[col].fillna(X_train[col].median())
    X_test[col] = X_test[col].fillna(X_train[col].median())

for col in categorical:
    X_train[col] = X_train[col].fillna(X_train[col].mode()[0])
    X_test[col] = X_test[col].fillna(X_train[col].mode()[0])

## 12. Label Encoding

In [ ]:
encoders = {}

for col in categorical:
    le = LabelEncoder()

    X_train[col] = le.fit_transform(X_train[col].astype(str))

    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    X_test[col] = X_test[col].astype(str).map(mapping).fillna(-1)

    encoders[col] = le

X_train.head()

## 13. Random Forest

In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Random Forest training completed.")

## 14. Model Evaluation

In [ ]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE :", round(mae, 2), "minutes")
print("RMSE:", round(rmse, 2), "minutes")
print("R2  :", round(r2, 3))

## 15. Actual vs Predicted

In [ ]:
plt.scatter(y_test, predictions, alpha=0.4)
plt.xlabel("Actual Delivery Time")
plt.ylabel("Predicted Delivery Time")
plt.title("Actual vs Predicted")
plt.show()

## 16. Feature Importance

In [ ]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance)

importance.head(10).plot(
    x="Feature",
    y="Importance",
    kind="bar"
)

plt.title("Top Important Features")
plt.show()

## 17. Create Predictions for Power BI

This creates simple prediction columns that can be used directly in Power BI.

In [ ]:
powerbi = df.loc[X_test.index].copy()

powerbi["Actual_Time"] = y_test.values
powerbi["Predicted_Time"] = predictions.round(1)

powerbi["Error"] = (
    powerbi["Predicted_Time"] - powerbi["Actual_Time"]
).round(1)

powerbi["Absolute_Error"] = powerbi["Error"].abs().round(1)

powerbi["Delay_Risk"] = pd.cut(
    powerbi["Predicted_Time"],
    bins=[0, 30, 40, np.inf],
    labels=["Low", "Medium", "High"]
)

powerbi.head()

## 18. Save Power BI Predictions

In [ ]:
powerbi.to_csv(
    "../data/Zomato_PowerBI_Predictions.csv",
    index=False
)

print("Zomato_PowerBI_Predictions.csv saved.")

## 19. Save Model Metrics

In [ ]:
metrics = pd.DataFrame({
    "Model": ["Random Forest"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2]
})

metrics.to_csv(
    "../data/Zomato_Model_Metrics.csv",
    index=False
)

metrics

# 20. Power BI Dashboard

Import these two files into Power BI:

### Zomato_PowerBI_Predictions.csv
Use it for:
- Total Orders
- Actual Delivery Time
- Predicted Delivery Time
- Delay Risk
- Traffic
- Weather
- City
- Distance

### Zomato_Model_Metrics.csv
Use it for:
- MAE
- RMSE
- R²

### Suggested Dashboard
**KPI Cards**
- Total Orders
- Average Delivery Time
- Average Predicted Time
- Average Error
- High Risk Orders

**Charts**
- Actual vs Predicted Delivery Time
- Orders by Delay Risk
- Average Delivery Time by Traffic
- Average Delivery Time by Weather
- Average Delivery Time by City
- Distance vs Delivery Time